# RAG_ICH – Notebook 02: Data Cleaning and Corpus Enrichment

## Objective

This notebook preprocesses the extracted Clinical Practice Guideline (CPG) for
Retrieval-Augmented Generation (RAG).

The preprocessing pipeline consists of:

1. Basic Cleaning
2. OCR Inspection and Correction
3. Duplicate Inspection
4. Visual Algorithm Enrichment
5. Validation

Output:
- cleaned_cpg_text.txt
- enriched_cpg_text.txt

In [ ]:
import re
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

INPUT_FILE = "/content/drive/MyDrive/RAG_ICH/data/processed/linearized_cpg_text.txt"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Loaded {len(text):,} characters.")

#1. Basic Cleaning

In [ ]:
def basic_cleaning(text):
    """
    Stage 1:
    - Unicode normalization
    - Whitespace cleanup
    - Bullet normalization
    """

    # Normalize quotation marks
    text = text.replace("“", '"')
    text = text.replace("”", '"')
    text = text.replace("’", "'")
    text = text.replace("‘", "'")

    # Normalize dashes
    text = text.replace("–", "-")
    text = text.replace("—", "-")

    # Normalize bullet symbols
    text = text.replace("•", "-")
    text = text.replace("◦", "-")
    text = text.replace("▪", "-")

    # Remove trailing spaces
    text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)

    # Collapse multiple blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text

In [ ]:
cleaned_text = basic_cleaning(text)

print(cleaned_text[:2000])

#2. OCR Correction

## OCR Correction Dictionary

In [ ]:
# ==========================================================
# Stage 2: OCR Correction Dictionary
# ==========================================================
# Dictionary of known OCR errors identified during inspection
# of the linearized Clinical Practice Guideline (CPG).
#
# Add new corrections here as more OCR errors are discovered.
# ==========================================================

OCR_CORRECTIONS = {

    # ------------------------------------------------------
    # Common OCR spelling errors
    # ------------------------------------------------------
    "anglograpny": "angiography",
    "cornputed": "computed",
    "haemorrnage": "haemorrhage",

    # ------------------------------------------------------
    # Roman numeral recognition errors
    # ------------------------------------------------------
    "1I": "II",
    "1l": "II",

    # ------------------------------------------------------
    # Missing spaces
    # ------------------------------------------------------
    "CTangiography": "CT angiography",
    "MRangiography": "MR angiography",

}

## OCR Correction Function

In [ ]:
import re

def correct_ocr(text, corrections):
    """
    Stage 2: Dictionary-based OCR correction.

    Parameters
    ----------
    text : str
        Input text.

    corrections : dict
        Dictionary of OCR corrections.

    Returns
    -------
    corrected_text : str
        OCR corrected text.

    correction_log : dict
        Detailed correction report.

    total_corrections : int
        Total number of corrections.
    """

    correction_log = {}
    total_corrections = 0

    # Split into lines so we know where corrections occur
    lines = text.splitlines()

    for wrong, correct in corrections.items():

        pattern = r"\b" + re.escape(wrong) + r"\b"

        occurrences = []

        for line_number, line in enumerate(lines, start=1):

            if re.search(pattern, line):

                count = len(re.findall(pattern, line))

                occurrences.append({
                    "line": line_number,
                    "text": line.strip(),
                    "count": count
                })

                lines[line_number - 1] = re.sub(pattern, correct, line)

                total_corrections += count

        if occurrences:

            correction_log[f"{wrong} → {correct}"] = occurrences

    corrected_text = "\n".join(lines)

    return corrected_text, correction_log, total_corrections

## Apply OCR Correction

In [ ]:
# Apply OCR correction
cleaned_text, correction_log, total_corrections = correct_ocr(
    cleaned_text,
    OCR_CORRECTIONS
)

## OCR Correction Report

In [ ]:
print("=" * 70)
print("OCR CORRECTION REPORT")
print("=" * 70)

if correction_log:

    for correction, occurrences in correction_log.items():

        total = sum(item["count"] for item in occurrences)

        print(f"\nCorrection : {correction}")
        print(f"Occurrences: {total}")

        for item in occurrences:

            print(f"  Line {item['line']}:")
            print(f"    {item['text']}")

        print("-" * 70)

else:

    print("No OCR corrections were applied.")

print(f"\nTotal OCR corrections: {total_corrections}")

 # 3. Duplicate Inspection/Removal

## Duplicate Removal Function

In [ ]:
def remove_consecutive_duplicates(text):
    """
    Stage 3: Remove consecutive duplicate lines while preserving
    legitimate repetitions occurring in different parts of the document.

    Parameters
    ----------
    text : str
        Input text.

    Returns
    -------
    cleaned_text : str
        Text with consecutive duplicate lines removed.

    removed_count : int
        Number of duplicate lines removed.

    removed_log : list
        Details of removed duplicate lines.
    """

    lines = text.splitlines()

    cleaned_lines = []
    removed_log = []

    previous_line = None
    removed_count = 0

    for line_number, line in enumerate(lines, start=1):

        current = line.strip()

        if current == previous_line and current != "":

            removed_log.append({
                "line": line_number,
                "text": current
            })

            removed_count += 1
            continue

        cleaned_lines.append(line)
        previous_line = current

    cleaned_text = "\n".join(cleaned_lines)

    return cleaned_text, removed_count, removed_log

## Run Duplicate Removal

In [ ]:
cleaned_text, duplicate_count, duplicate_log = remove_consecutive_duplicates(cleaned_text)

## Summary

In [ ]:
print("=" * 70)
print("DUPLICATE REMOVAL REPORT")
print("=" * 70)

if duplicate_log:

    for item in duplicate_log:
        print(f"Line {item['line']}: {item['text']}")

else:
    print("No consecutive duplicate lines found.")

print("-" * 70)
print(f"Duplicate lines removed: {duplicate_count}")

In [ ]:
from collections import Counter

lines = [line.strip() for line in cleaned_text.splitlines() if line.strip()]

line_counts = Counter(lines)

duplicates = {line: count for line, count in line_counts.items() if count > 1}

print(f"Unique lines: {len(line_counts)}")
print(f"Duplicated lines: {len(duplicates)}")

In [ ]:
for line, count in sorted(duplicates.items(), key=lambda x: x[1], reverse=True)[:30]:
    print(f"{count}x  {line}")

In [ ]:
# ==========================================================
# Save Cleaned CPG Text
# ==========================================================

OUTPUT_FILE = "/content/drive/MyDrive/RAG_ICH/data/processed/cleaned_cpg_text.txt"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print(f"Cleaned CPG saved successfully to:\n{OUTPUT_FILE}")

##4. Visual Algorithm Enrichment

Prompt used in Gemini: You are assisting with the preprocessing of a Clinical Practice Guideline for a Retrieval-Augmented Generation (RAG) system.

The attached page contains a clinical decision algorithm.

Your task is to convert the algorithm into structured text.

Requirements:

Preserve every clinical decision.
Preserve all conditions.
Preserve every recommendation.
Preserve all abbreviations.
Preserve the decision order.
Do not summarise.
Do not interpret.
Do not omit branches.
Format as numbered decision steps.
Output only the structured algorithm.

In [ ]:
# ==========================================================
# Stage 4: Load Algorithm Enrichment Files
# ==========================================================

ALGORITHM_1 = "/content/drive/MyDrive/RAG_ICH/data/processed/algorithm_1_neuroimaging.txt"
ALGORITHM_2 = "/content/drive/MyDrive/RAG_ICH/data/processed/algorithm_2_neurosurgery.txt"

with open(ALGORITHM_1, "r", encoding="utf-8") as f:
    algorithm1 = f.read()

with open(ALGORITHM_2, "r", encoding="utf-8") as f:
    algorithm2 = f.read()

print("Algorithm files loaded successfully.")

In [ ]:
# ==========================================================
# Stage 4: Merge Algorithm Enrichment
# ==========================================================

enriched_text = (
    cleaned_text
    + "\n\n"
    + algorithm1
    + "\n\n"
    + algorithm2
)

print("Algorithms successfully merged into the corpus.")

In [ ]:
print("=" * 60)

print(f"Original corpus size : {len(cleaned_text):,} characters")
print(f"Enriched corpus size : {len(enriched_text):,} characters")

print("=" * 60)

In [ ]:
assert "ALGORITHM 1" in enriched_text
assert "ALGORITHM 2" in enriched_text

print("Algorithm enrichment verified.")

In [ ]:
OUTPUT_FILE = "/content/drive/MyDrive/RAG_ICH/data/processed/enriched_cpg_text.txt"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(enriched_text)

print(f"Saved enriched corpus to:\n{OUTPUT_FILE}")

# 5. Validation

## 5.1. Corpus Statistics

In [ ]:
print("=" * 60)
print("CORPUS STATISTICS")
print("=" * 60)

print(f"Original text length : {len(text):,} characters")
print(f"Cleaned text length  : {len(cleaned_text):,} characters")
print(f"Enriched text length : {len(enriched_text):,} characters")

## 5.2. Algorithm Verification

In [ ]:
print("=" * 60)
print("ALGORITHM VERIFICATION")
print("=" * 60)

algorithms = [
    "ALGORITHM 1",
    "ALGORITHM 2"
]

for alg in algorithms:
    if alg in enriched_text:
        print(f"✓ {alg} found")
    else:
        print(f"✗ {alg} NOT found")

## 5.3. Section Verification

In [ ]:
sections = [
    "Introduction",
    "Diagnosis",
    "Acute Medical Management",
    "Neurosurgery",
    "Intensive Care",
    "Rehabilitation",
    "Secondary Prevention"
]

print("=" * 60)
print("SECTION VERIFICATION")
print("=" * 60)

for section in sections:

    if section.lower() in enriched_text.lower():
        print(f"✓ {section}")
    else:
        print(f"⚠ {section} not found")

## 5.4. File Statistics

In [ ]:
words = len(enriched_text.split())
lines = len(enriched_text.splitlines())

print("=" * 60)
print("FILE STATISTICS")
print("=" * 60)

print(f"Words : {words:,}")
print(f"Lines : {lines:,}")